# Last 10 Bitcoin Price Predictions Comparison
This notebook compares the last 10 price predictions (separated by commas) from both the enhanced and base models.

## Install Required Libraries

In [ ]:
!pip install transformers datasets torch peft accelerate matplotlib seaborn pandas numpy

## Load Required Libraries and Models

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datasets import load_dataset
import json
import re

# Load the enhanced model
base_model_id = './Qwen3-8B'
adapter_path = './my-awesome-model_final_bitcoin-enhanced-prediction-dataset-with-local-comprehensive-news-v2/checkpoint-400'

# Load the base model and tokenizer
base_qwen_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    adapter_path,
    trust_remote_code=True
)

base_qwen_tokenizer = tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_qwen_model.resize_token_embeddings(len(tokenizer))

# Load the enhanced model with LoRA adapter
enhanced_model = PeftModel.from_pretrained(base_qwen_model, adapter_path)
enhanced_model.eval()

print("Models loaded successfully!")

## Load Test Dataset

In [ ]:
# Load the enhanced dataset
test_dataset = load_dataset('tahamajs/bitcoin-enhanced-prediction-dataset-with-local-comprehensive-news', split='train')
print(f"Loaded {len(test_dataset)} test samples")

## Utility Functions for Price Extraction

In [ ]:
def extract_trading_recommendation_from_text(text):
    """Extract structured trading recommendation with 10-day forecast from model output"""
    import json
    
    # Try to find JSON-like structure in the text
    json_pattern = r'\{[^{}]*"forecast_10d"[^{}]*\}'
    matches = re.findall(json_pattern, text, re.DOTALL)
    
    if matches:
        for match in matches:
            try:
                # Clean up the JSON string
                cleaned_json = match.strip()
                parsed = json.loads(cleaned_json)
                
                # Validate required fields
                if all(key in parsed for key in ['action', 'confidence', 'forecast_10d']):
                    return parsed
            except json.JSONDecodeError:
                continue
    
    # Fallback: try to extract forecast_10d array separately
    forecast_pattern = r'"forecast_10d"\s*:\s*\[([^\]]+)\]'
    forecast_match = re.search(forecast_pattern, text)
    
    if forecast_match:
        try:
            price_str = forecast_match.group(1)
            prices = [float(p.strip()) for p in price_str.split(',')]
            return {
                'action': 'UNKNOWN',
                'confidence': 0,
                'forecast_10d': prices,
                'stop_loss': None,
                'take_profit': None
            }
        except:
            pass
    
    # Last resort: extract comma-separated numbers
    price_pattern = r'(\d+(?:\.\d+)?(?:,\s*\d+(?:\.\d+)?)*)'  
    matches = re.findall(price_pattern, text)
    
    if matches:
        longest_match = max(matches, key=len)
        try:
            prices = [float(p.strip()) for p in longest_match.split(',')]
            return {
                'action': 'UNKNOWN',
                'confidence': 0,
                'forecast_10d': prices[-10:] if len(prices) >= 10 else prices,
                'stop_loss': None,
                'take_profit': None
            }
        except:
            pass
    
    return None

def extract_last_10_prices_from_text(text):
    """Extract the last 10 price predictions from model output (backward compatibility)"""
    recommendation = extract_trading_recommendation_from_text(text)
    if recommendation and 'forecast_10d' in recommendation:
        return recommendation['forecast_10d']
    return []

def extract_all_prices_from_text(text):
    """Extract all price predictions from model output (backward compatibility)"""
    return extract_last_10_prices_from_text(text)

def format_input_enhanced(example):
    """Format input for the enhanced model"""
    instruction = example.get('instruction', '')
    user_input = example.get('input', '')
    messages = [
        {'role': 'system', 'content': instruction},
        {'role': 'user', 'content': user_input}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def format_input_base(example):
    """Format input for base Qwen model"""
    instruction = example.get('instruction', '')
    user_input = example.get('input', '')
    
    bitcoin_instruction = """You are a Bitcoin investment advisor. Based on the provided market data and news, predict the next 10 days of Bitcoin prices. Provide your predictions as comma-separated numbers."""
    
    messages = [
        {'role': 'system', 'content': bitcoin_instruction},
        {'role': 'user', 'content': f"{instruction}\n\n{user_input}\n\nPlease provide 10 Bitcoin price predictions for the next 10 days, separated by commas."}
    ]
    return base_qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Utility functions loaded!")

## Generate Last 10 Price Predictions

In [ ]:
# Generate predictions for multiple samples
num_samples = 20  # Number of test samples to analyze
results = []

print(f"Generating last 10 price predictions for {num_samples} samples...\n")

for i in range(min(num_samples, len(test_dataset))):
    test_example = test_dataset[i]
    sample_result = {
        'sample_id': i,
        'enhanced_last_10': [],
        'base_last_10': [],
        'actual_last_10': [],
        'enhanced_full': [],
        'base_full': [],
        'actual_full': []
    }
    
    # Enhanced Model Prediction
    enhanced_text = format_input_enhanced(test_example)
    enhanced_inputs = tokenizer(enhanced_text, return_tensors='pt', truncation=True, max_length=2048)
    enhanced_inputs = {k: v.to(enhanced_model.device) for k, v in enhanced_inputs.items()}
    
    with torch.no_grad():
        enhanced_outputs = enhanced_model.generate(
            **enhanced_inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    enhanced_generated = tokenizer.decode(enhanced_outputs[0][enhanced_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Base Model Prediction
    base_text = format_input_base(test_example)
    base_inputs = base_qwen_tokenizer(base_text, return_tensors='pt', truncation=True, max_length=2048)
    base_inputs = {k: v.to(base_qwen_model.device) for k, v in base_inputs.items()}
    
    with torch.no_grad():
        base_outputs = base_qwen_model.generate(
            **base_inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=base_qwen_tokenizer.eos_token_id
        )
    
    base_generated = base_qwen_tokenizer.decode(base_outputs[0][base_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Extract prices
    enhanced_prices_full = extract_all_prices_from_text(enhanced_generated)
    base_prices_full = extract_all_prices_from_text(base_generated)
    actual_prices_full = extract_all_prices_from_text(test_example.get('output', ''))
    
    enhanced_prices_last_10 = extract_last_10_prices_from_text(enhanced_generated)
    base_prices_last_10 = extract_last_10_prices_from_text(base_generated)
    actual_prices_last_10 = extract_last_10_prices_from_text(test_example.get('output', ''))
    
    # Store results
    sample_result.update({
        'enhanced_last_10': enhanced_prices_last_10,
        'base_last_10': base_prices_last_10,
        'actual_last_10': actual_prices_last_10,
        'enhanced_full': enhanced_prices_full,
        'base_full': base_prices_full,
        'actual_full': actual_prices_full,
        'enhanced_text': enhanced_generated,
        'base_text': base_generated
    })
    
    results.append(sample_result)
    
    # Print progress
    print(f"Sample {i+1}:")
    print(f"  Enhanced Last 10: {enhanced_prices_last_10}")
    print(f"  Base Last 10:     {base_prices_last_10}")
    print(f"  Actual Last 10:   {actual_prices_last_10}")
    print(f"  Enhanced Count: {len(enhanced_prices_full)}, Base Count: {len(base_prices_full)}, Actual Count: {len(actual_prices_full)}")
    print("-" * 80)

print(f"\n✅ Generated predictions for {len(results)} samples!")

## Analysis of Last 10 Prices

In [ ]:
# Analyze the last 10 prices
valid_comparisons = []

for result in results:
    if (len(result['enhanced_last_10']) >= 5 and 
        len(result['base_last_10']) >= 5 and 
        len(result['actual_last_10']) >= 5):
        valid_comparisons.append(result)

print(f"Found {len(valid_comparisons)} samples with valid last 10 price predictions\n")

if valid_comparisons:
    # Calculate statistics for last 10 prices
    enhanced_last_10_all = []
    base_last_10_all = []
    actual_last_10_all = []
    
    for comp in valid_comparisons:
        enhanced_last_10_all.extend(comp['enhanced_last_10'])
        base_last_10_all.extend(comp['base_last_10'])
        actual_last_10_all.extend(comp['actual_last_10'])
    
    print("=" * 80)
    print("📊 LAST 10 PRICES STATISTICS")
    print("=" * 80)
    
    # Enhanced model stats
    print(f"\n🔥 ENHANCED MODEL LAST 10 PRICES:")
    print(f"  Mean: ${np.mean(enhanced_last_10_all):.2f}")
    print(f"  Median: ${np.median(enhanced_last_10_all):.2f}")
    print(f"  Std: ${np.std(enhanced_last_10_all):.2f}")
    print(f"  Min: ${np.min(enhanced_last_10_all):.2f}")
    print(f"  Max: ${np.max(enhanced_last_10_all):.2f}")
    print(f"  Total predictions: {len(enhanced_last_10_all)}")
    
    # Base model stats
    print(f"\n⚡ BASE MODEL LAST 10 PRICES:")
    print(f"  Mean: ${np.mean(base_last_10_all):.2f}")
    print(f"  Median: ${np.median(base_last_10_all):.2f}")
    print(f"  Std: ${np.std(base_last_10_all):.2f}")
    print(f"  Min: ${np.min(base_last_10_all):.2f}")
    print(f"  Max: ${np.max(base_last_10_all):.2f}")
    print(f"  Total predictions: {len(base_last_10_all)}")
    
    # Actual prices stats
    print(f"\n🎯 ACTUAL LAST 10 PRICES:")
    print(f"  Mean: ${np.mean(actual_last_10_all):.2f}")
    print(f"  Median: ${np.median(actual_last_10_all):.2f}")
    print(f"  Std: ${np.std(actual_last_10_all):.2f}")
    print(f"  Min: ${np.min(actual_last_10_all):.2f}")
    print(f"  Max: ${np.max(actual_last_10_all):.2f}")
    print(f"  Total actual prices: {len(actual_last_10_all)}")
    
    # Sample-by-sample comparison
    print(f"\n📋 SAMPLE-BY-SAMPLE LAST 10 PRICES COMPARISON:")
    print("-" * 80)
    
    for i, comp in enumerate(valid_comparisons[:10]):  # Show first 10 samples
        print(f"\nSample {comp['sample_id']+1}:")
        print(f"  Enhanced: {[f'${p:.2f}' for p in comp['enhanced_last_10']]}")
        print(f"  Base:     {[f'${p:.2f}' for p in comp['base_last_10']]}")
        print(f"  Actual:   {[f'${p:.2f}' for p in comp['actual_last_10']]}")
        
        # Calculate differences
        if len(comp['actual_last_10']) > 0:
            min_len = min(len(comp['enhanced_last_10']), len(comp['actual_last_10']))
            if min_len > 0:
                enhanced_diff = np.mean(np.abs(np.array(comp['enhanced_last_10'][:min_len]) - np.array(comp['actual_last_10'][:min_len])))
                base_diff = np.mean(np.abs(np.array(comp['base_last_10'][:min_len]) - np.array(comp['actual_last_10'][:min_len])))
                print(f"  Enhanced MAE: ${enhanced_diff:.2f}")
                print(f"  Base MAE:     ${base_diff:.2f}")
                print(f"  Better model: {'Enhanced' if enhanced_diff < base_diff else 'Base'}")

else:
    print("❌ No valid comparisons found with sufficient last 10 price predictions")

## Visualization of Last 10 Prices

In [ ]:
if valid_comparisons:
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Last 10 Bitcoin Price Predictions Comparison', fontsize=16, fontweight='bold')
    
    # 1. Distribution comparison of last 10 prices
    axes[0,0].hist([enhanced_last_10_all, base_last_10_all, actual_last_10_all], 
                   bins=30, alpha=0.7, 
                   label=['Enhanced Model', 'Base Model', 'Actual Prices'], 
                   color=['blue', 'red', 'green'])
    axes[0,0].set_title('Distribution of Last 10 Price Predictions', fontweight='bold')
    axes[0,0].set_xlabel('Price ($)')
    axes[0,0].set_ylabel('Frequency')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Box plot comparison
    data_for_boxplot = [enhanced_last_10_all, base_last_10_all, actual_last_10_all]
    labels_for_boxplot = ['Enhanced', 'Base', 'Actual']
    axes[0,1].boxplot(data_for_boxplot, labels=labels_for_boxplot)
    axes[0,1].set_title('Last 10 Prices Box Plot Comparison', fontweight='bold')
    axes[0,1].set_ylabel('Price ($)')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Sample-wise comparison for first few samples
    sample_indices = list(range(min(5, len(valid_comparisons))))
    enhanced_means = [np.mean(valid_comparisons[i]['enhanced_last_10']) for i in sample_indices]
    base_means = [np.mean(valid_comparisons[i]['base_last_10']) for i in sample_indices]
    actual_means = [np.mean(valid_comparisons[i]['actual_last_10']) for i in sample_indices]
    
    x = np.arange(len(sample_indices))
    width = 0.25
    
    axes[1,0].bar(x - width, enhanced_means, width, label='Enhanced', alpha=0.8, color='blue')
    axes[1,0].bar(x, base_means, width, label='Base', alpha=0.8, color='red')
    axes[1,0].bar(x + width, actual_means, width, label='Actual', alpha=0.8, color='green')
    
    axes[1,0].set_title('Mean of Last 10 Prices by Sample', fontweight='bold')
    axes[1,0].set_xlabel('Sample Index')
    axes[1,0].set_ylabel('Mean Price ($)')
    axes[1,0].set_xticks(x)
    axes[1,0].set_xticklabels([f'S{valid_comparisons[i]["sample_id"]+1}' for i in sample_indices])
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Price trend for a specific sample
    if len(valid_comparisons) > 0:
        sample_to_plot = valid_comparisons[0]
        days = list(range(1, len(sample_to_plot['enhanced_last_10']) + 1))
        
        axes[1,1].plot(days, sample_to_plot['enhanced_last_10'], 'o-', label='Enhanced', color='blue', linewidth=2)
        axes[1,1].plot(days, sample_to_plot['base_last_10'][:len(days)], 's-', label='Base', color='red', linewidth=2)
        axes[1,1].plot(days, sample_to_plot['actual_last_10'][:len(days)], '^-', label='Actual', color='green', linewidth=2)
        
        axes[1,1].set_title(f'Price Trend - Sample {sample_to_plot["sample_id"]+1}', fontweight='bold')
        axes[1,1].set_xlabel('Day')
        axes[1,1].set_ylabel('Price ($)')
        axes[1,1].legend()
        axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('last_10_prices_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("📊 Last 10 prices comparison visualization saved as 'last_10_prices_comparison.png'")

else:
    print("❌ Cannot create visualizations - insufficient data")

## Detailed Last 10 Prices Analysis

In [ ]:
if valid_comparisons:
    # Calculate MAE for last 10 prices only
    enhanced_mae_last_10 = []
    base_mae_last_10 = []
    
    for comp in valid_comparisons:
        # Enhanced vs Actual
        min_len_enhanced = min(len(comp['enhanced_last_10']), len(comp['actual_last_10']))
        if min_len_enhanced > 0:
            enhanced_mae = np.mean(np.abs(np.array(comp['enhanced_last_10'][:min_len_enhanced]) - 
                                        np.array(comp['actual_last_10'][:min_len_enhanced])))
            enhanced_mae_last_10.append(enhanced_mae)
        
        # Base vs Actual
        min_len_base = min(len(comp['base_last_10']), len(comp['actual_last_10']))
        if min_len_base > 0:
            base_mae = np.mean(np.abs(np.array(comp['base_last_10'][:min_len_base]) - 
                                    np.array(comp['actual_last_10'][:min_len_base])))
            base_mae_last_10.append(base_mae)
    
    print("=" * 80)
    print("🎯 LAST 10 PRICES ACCURACY ANALYSIS")
    print("=" * 80)
    
    if enhanced_mae_last_10 and base_mae_last_10:
        print(f"\n📊 Mean Absolute Error (MAE) for Last 10 Prices:")
        print(f"  Enhanced Model MAE: ${np.mean(enhanced_mae_last_10):.2f} (±${np.std(enhanced_mae_last_10):.2f})")
        print(f"  Base Model MAE:     ${np.mean(base_mae_last_10):.2f} (±${np.std(base_mae_last_10):.2f})")
        
        improvement = ((np.mean(base_mae_last_10) - np.mean(enhanced_mae_last_10)) / np.mean(base_mae_last_10)) * 100
        print(f"  Improvement: {improvement:.2f}%")
        
        # Count how many times enhanced model is better
        enhanced_better_count = sum(1 for e, b in zip(enhanced_mae_last_10, base_mae_last_10) if e < b)
        print(f"\n🏆 Enhanced model is more accurate in {enhanced_better_count}/{len(enhanced_mae_last_10)} samples ({enhanced_better_count/len(enhanced_mae_last_10)*100:.1f}%)")
    
    # Save detailed results
    detailed_results = {
        'summary': {
            'total_samples_analyzed': len(valid_comparisons),
            'enhanced_model_mae_last_10': float(np.mean(enhanced_mae_last_10)) if enhanced_mae_last_10 else None,
            'base_model_mae_last_10': float(np.mean(base_mae_last_10)) if base_mae_last_10 else None,
            'improvement_percentage': float(improvement) if enhanced_mae_last_10 and base_mae_last_10 else None,
            'enhanced_better_count': enhanced_better_count if enhanced_mae_last_10 and base_mae_last_10 else None
        },
        'detailed_comparisons': []
    }
    
    for comp in valid_comparisons:
        detailed_results['detailed_comparisons'].append({
            'sample_id': comp['sample_id'],
            'enhanced_last_10': [float(p) for p in comp['enhanced_last_10']],
            'base_last_10': [float(p) for p in comp['base_last_10']],
            'actual_last_10': [float(p) for p in comp['actual_last_10']],
            'enhanced_generated_text': comp['enhanced_text'],
            'base_generated_text': comp['base_text']
        })
    
    with open('last_10_prices_detailed_analysis.json', 'w') as f:
        json.dump(detailed_results, f, indent=2)
    
    print(f"\n💾 Detailed last 10 prices analysis saved to 'last_10_prices_detailed_analysis.json'")

else:
    print("❌ No valid data for detailed analysis")

## Show Raw Model Outputs for Last Few Samples

In [ ]:
if results:
    print("🔍 RAW MODEL OUTPUTS (Last 3 Samples):")
    print("=" * 100)
    
    for i, result in enumerate(results[-3:]):  # Show last 3 samples
        print(f"\n📋 SAMPLE {result['sample_id']+1}:")
        print("-" * 50)
        
        print(f"\n🔥 ENHANCED MODEL OUTPUT:")
        print(f"Text: {result['enhanced_text']}")
        print(f"Extracted Prices: {result['enhanced_full']}")
        print(f"Last 10: {result['enhanced_last_10']}")
        
        print(f"\n⚡ BASE MODEL OUTPUT:")
        print(f"Text: {result['base_text']}")
        print(f"Extracted Prices: {result['base_full']}")
        print(f"Last 10: {result['base_last_10']}")
        
        print(f"\n🎯 ACTUAL OUTPUT:")
        print(f"Extracted Prices: {result['actual_full']}")
        print(f"Last 10: {result['actual_last_10']}")
        
        print("=" * 100)

else:
    print("❌ No results to display")